In [ ]:
import torch
import torch.nn as nn

class CausalAttentionExercise(nn.Module):
    def __init__(self, d_in, d_out, context_length):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        
        # [문제 1-1] 대각선 위쪽(미래 시점)이 1로 채워진 상삼각 행렬 마스크를 생성하여 등록하세요.
        # (힌트: torch.triu와 diagonal 인자를 사용하세요)
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)

        # [문제 1-2] 생성한 마스크를 사용하여 미래 토큰 위치를 -inf로 채우세요. (masked_fill_ 사용)
        # (힌트: 입력 x의 실제 토큰 개수만큼 마스크 크기를 조절해야 합니다)
        # 여기에 코드를 작성하세요
        attn_scores.masked_fill_(self.mask[:num_tokens, :num_tokens] == 1, float('-inf'))

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec